# LED types, drive circuits and speed {#sec-drive}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter3/notebooks/led-drive.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter3/notebooks/led-drive.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter3/semilib.py', 'semilib.py')
else:
    sys.path.insert(0, '..')                 # semilib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import brentq
import semilib as sl
sl.use_style()

*Slides 134–138 (surface- and edge-emitting LEDs, coupling into fibers, pigtailed LEDs) and 144–151 (analog drive, driver
modules, sinusoidal modulation and bandwidth, digital drive, rise and fall times, speed-up capacitors).*

## Key relations

The light output follows the current, so an LED is driven by a **current source**. The simplest driver is a resistor in series with
a voltage supply, $I_F=(V-V_F)/R$ (slide 144). For a transistor or op-amp current source $I_F\approx V/R_F$.

Modulating the current at frequency $f$ modulates the injected carrier population. Carriers disappear with the net lifetime
$\tau$, where $1/\tau=1/\tau_r+1/\tau_{nr}$, so the LED behaves like a first-order low-pass filter (slides 146–147):

$$
\frac{P_o(\omega)}{P_o(0)}=\frac{1}{\sqrt{1+(\omega\tau)^2}},\qquad
f_c=\frac{1}{2\pi\tau}\ \text{(electrical bandwidth)},\qquad f_\text{op}=\frac{\sqrt3}{2\pi\tau}\ \text{(optical bandwidth)}.
$$

At $f_c$ the optical power drops to 0.707, so the detected electrical power ($\propto I_\text{ph}^2$) drops to 1/2. At $f_\text{op}$
the optical power itself drops to 1/2. For digital modulation the same time constant gives 10–90 % rise and fall times
(slide 148)

$$
\tau_R=\tau_F=2.2\,\tau .
$$

## Modulation explorer

A 16-bit NRZ pattern drives the LED. The **speed-up capacitor** of slide 151 adds a current spike at each transition that decays
with the time constant $RC$, chosen here equal to $\tau$. Its strength $k$ is the ratio of the spike to the step.

```{ojs}
//| echo: false
viewof md = Inputs.form({
  tau: Inputs.range([0.5, 50], {value: 10, step: 0.5, label: "carrier lifetime τ (ns)"}),
  B: Inputs.range([5, 500], {value: 50, step: 1, label: "bit rate (Mb/s)"}),
  k: Inputs.range([0, 1.5], {value: 0, step: 0.05, label: "speed-up strength k (RC = τ)"})
})
```

```{ojs}
//| echo: false
mdOut = {
  const bits = [0,1,0,1,1,0,0,1,1,1,0,1,0,0,0,1], Tb = 1000/md.B, tau = md.tau;   // ns
  const N = 16*Tb, dt = Math.min(Tb, tau)/200, data = [];
  let n = 0, last = 0, tTr = -1e9, step = 0;
  for (let t = 0; t < N; t += dt) {
    const b = bits[Math.min(15, Math.floor(t/Tb))];
    if (b !== last) { step = b - last; tTr = t; last = b; }
    const I = b + md.k*step*Math.exp(-(t - tTr)/tau);
    n += dt*(I - n)/tau;                         // dN/dt = I/e − N/τ (normalised)
    if (data.length === 0 || t - data[data.length - 1].t > N/1500) data.push({t, P: n, I: b});
  }
  return {data, fc: 1000/(2*Math.PI*tau), fop: 1000*Math.sqrt(3)/(2*Math.PI*tau), tr: 2.2*tau, Tb};
}
Plot.plot({
  width: Math.min(width, 1050), height: 280, x: {label: "time (ns)"}, y: {label: "normalised light output", grid: true, domain: [-0.1, 1.3]},
  marks: [
    Plot.line(mdOut.data, {x: "t", y: "I", stroke: "#52514e", strokeWidth: 1.2, strokeDasharray: "4,3", curve: "step-after"}),
    Plot.line(mdOut.data, {x: "t", y: "P", stroke: "#2a78d6", strokeWidth: 2.2})
  ]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">f<sub>c</sub> (electrical, −3 dB) = <b>${mdOut.fc.toFixed(1)} MHz</b> · f<sub>op</sub> (optical, half power) = ${mdOut.fop.toFixed(1)} MHz ·
rise time 2.2τ = <b>${mdOut.tr.toFixed(1)} ns</b> · bit slot = ${mdOut.Tb.toFixed(1)} ns
${mdOut.tr > 0.7*mdOut.Tb ? html`<br><b>Rise time longer than ~70 % of the bit slot: the eye closes (intersymbol interference).</b>` : ""}</div>`
```

Set $k=1$: the spike exactly cancels the carrier pole and the optical output becomes a perfect copy of the drive pattern. In
practice the cancellation is limited by the driver's current headroom and by the junction capacitance (@sec-smallsignal). Too
much speed-up ($k>1$) produces overshoot.

## Frequency response (slides 146–147)

In [ ]:
#| label: fig-response
#| fig-cap: "LED modulation response for τ = 5 ns. Optical power (left axis) and detected electrical power ∝ I_ph² (dashed). The electrical −3 dB point f_c = 1/(2πτ) and the optical half-power point f_op = √3/(2πτ) differ by √3 (slide 146)."
tau = 5e-9
f = np.logspace(5, 9.5, 500)
R = sl.led_response(f, tau)
fc, fop = 1/(2*np.pi*tau), np.sqrt(3)/(2*np.pi*tau)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.semilogx(f, R, label="optical power P_o(f)/P_o(0)")
ax.semilogx(f, R**2, "--", label="electrical power [I_ph(f)/I_ph(0)]²")
for x, lab in ((fc, "f_c"), (fop, "f_op")):
    ax.axvline(x, color=sl.INK2, lw=1, ls=":"); ax.text(x*1.05, 0.9, f"{lab} = {x/1e6:.1f} MHz", fontsize=8)
ax.axhline(0.5, color=sl.GRID, lw=1.5); ax.axhline(1/np.sqrt(2), color=sl.GRID, lw=1.5)
ax.set(xlabel="modulation frequency (Hz)", ylabel="normalised response"); ax.legend(fontsize=8, loc="lower left"); plt.show()

In [ ]:
#| label: tbl-speed
#| tbl-cap: "Bandwidth and rise time for typical LED lifetimes. NRZ bit rate estimated as 0.7/t_r (a common rule of thumb)."
pd.DataFrame({f"τ = {t*1e9:g} ns": {"f_c (MHz)": 1/(2*np.pi*t)/1e6, "f_op (MHz)": np.sqrt(3)/(2*np.pi*t)/1e6,
              "t_r = 2.2τ (ns)": 2.2*t*1e9, "NRZ ≈ 0.7/t_r (Mb/s)": 0.7/(2.2*t)/1e6}
              for t in (100e-9, 20e-9, 5e-9, 1e-9, 0.3e-9)}).T.round(1)

Visible-light LEDs (τ ≈ 10–50 ns) reach tens of MHz, enough for Li-Fi with OFDM. Heavily doped IR communication LEDs and
microLEDs (τ < 1 ns) reach hundreds of MHz to a few GHz, which makes them candidates for short optical interconnects.

## Rise time and the speed-up capacitor (slides 148–151)

In [ ]:
#| label: fig-speedup
#| fig-cap: "Response to a 50 ns current pulse (τ = 5 ns). Without pulse shaping the 10–90 % rise time is 2.2τ = 11 ns. A speed-up capacitor with RC = τ (k = 1) cancels the pole; k = 0.5 is a partial correction."
t = np.linspace(-10e-9, 100e-9, 4000); dt = t[1] - t[0]
tau, t_on, t_off = 5e-9, 0.0, 50e-9
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for k in (0.0, 0.5, 1.0):
    I = ((t >= t_on) & (t < t_off)).astype(float)
    I += k*np.where(t >= t_on, np.exp(-(t - t_on)/tau), 0)*(t < t_off) - k*np.where(t >= t_off, np.exp(-(t - t_off)/tau), 0)
    n = np.zeros_like(t)
    for i in range(1, t.size): n[i] = n[i-1] + dt*(I[i-1] - n[i-1])/tau
    ax.plot(t*1e9, n, label=f"k = {k}")
    if k == 0:
        r = t[(n >= 0.1) & (t < t_off)]; r9 = t[(n >= 0.9) & (t < t_off)]
        tr = r9[0] - r[0]
ax.plot(t*1e9, ((t >= t_on) & (t < t_off)), ":", color=sl.INK2, lw=1.2, label="drive")
ax.set(xlabel="t (ns)", ylabel="normalised light output"); ax.legend(fontsize=8); plt.show()
print(f"10–90 % rise time without shaping: {tr*1e9:.2f} ns = {tr/tau:.2f} τ")

## Choosing the bias resistor (slide 144)

The LED I–V curve is steep, so a voltage source alone would be unstable: a 50 mV change gives a large current change. The
series resistor sets the operating point. With the diode model of @sec-diode (including series resistance) the exact current
is the root of $V_s=IR+V_F(I)$:

In [ ]:
#| label: tbl-resistor
#| tbl-cap: "Red LED (η = 2, I_o = 10⁻¹⁸ A, R_s = 8 Ω, a typical fit to a 5 mm indicator LED) driven from 5 V. Target 20 mA: the simple formula R = (V_s − V_F)/I_F with the datasheet V_F is accurate because R ≫ r_d."
Io, eta, Rs, Vs = 1e-18, 2.0, 8.0, 5.0
VF = lambda I: eta*sl.kT()*np.log(I/Io + 1) + I*Rs
rows = {}
for R in (100, 150, 220, 330):
    I = brentq(lambda I: Vs - I*R - VF(I), 1e-9, Vs/R)
    rows[f"R = {R} Ω"] = {"I_F (mA)": I*1e3, "V_F (V)": VF(I), "P_LED (mW)": I*VF(I)*1e3, "P_R (mW)": I**2*R*1e3,
                          "dI/dV_s (mA/V)": 1e3/(R + Rs + eta*sl.kT()/I)}
print(f"V_F at 20 mA = {VF(0.02):.3f} V  ->  R = (5 − V_F)/0.02 = {(Vs - VF(0.02))/0.02:.0f} Ω")
pd.DataFrame(rows).T.round(2)

## Coupling LEDs to fibers (slides 135–138)

A surface-emitting LED is close to a **Lambertian** source, $I(\theta)\propto\cos\theta$. A fiber accepts only rays inside its
numerical aperture, so the coupled fraction from a small source is $\eta_c=\text{NA}^2$. A source larger than the core loses a
further area factor $(a/r_s)^2$. Edge emitters (ELEDs) have a narrower beam in one plane and couple better.

In [ ]:
#| label: tbl-coupling
#| tbl-cap: "Coupling efficiency of a Lambertian LED (emitting diameter 50 µm) into common fibers, η = NA² × min(1, (core/source)²), plus a Fresnel loss of a 1.5-index interface."
fibers = {"SMF-28 (9 µm, NA 0.14)": (9, 0.14), "MMF 50/125 (NA 0.20)": (50, 0.20),
          "MMF 62.5/125 (NA 0.275)": (62.5, 0.275), "POF 980/1000 (NA 0.50)": (980, 0.50)}
src = 50
pd.DataFrame({k: {"η geometric": f"{NA**2*min(1, (d/src)**2)*100:.2f} %",
                  "η with Fresnel (×0.96)": f"{NA**2*min(1, (d/src)**2)*0.96*100:.2f} %",
                  "loss (dB)": round(-10*np.log10(NA**2*min(1, (d/src)**2)*0.96), 1)} for k, (d, NA) in fibers.items()}).T

A 1 mW LED puts only about 1 µW into a single-mode fiber (–30 dB), but 40 µW into a 50 µm multimode fiber and 250 µW into a
plastic optical fiber. LEDs were therefore used with multimode and plastic fibers (short links, home networks, automotive),
while single-mode links use lasers (Ch. 4).

::: {.callout-tip collapse="true"}
## Going deeper
- **Rate equation with carrier-dependent lifetime.** With $1/\tau=A+BN$ the rise and fall become asymmetric: the turn-off tail is
  slower. Integrate $dN/dt=I/eV-AN-BN^2$ and compare with the linear model.
- **Bias and small-signal modulation.** Driving around a DC bias $I_0$ shortens the differential lifetime $\tau_\text{diff}=1/(A+2BN_0)$:
  the bandwidth grows with the bias. Plot $f_c$ against $I_0$.
- **Visible light communication.** With $f_c\approx20$ MHz, OFDM with bit loading reaches several Gb/s on a single LED. Estimate
  the Shannon capacity $\int\log_2(1+\text{SNR}(f))\,df$ with the first-order response and a white-noise receiver.
- **PWM dimming.** Driver modules (slide 145) dim by pulse-width modulation. How fast must the PWM be to avoid visible flicker, and
  why is it not limited by $\tau$?
:::

::: {.callout-note collapse="true"}
## Complements
- Kasap, *Optoelectronics and Photonics*, sec. 3.16–3.17; Schubert, *Light-Emitting Diodes*, ch. 24 (high-speed LEDs).
- Texas Instruments and Analog Devices application notes on LED drivers (constant-current topologies).
- H. Haas *et al.*, "What is LiFi?", *J. Lightwave Technol.* 34, 1533 (2016).
:::

## Try it yourself

1. An LED has $\tau_r=20$ ns and $\tau_{nr}=80$ ns. Find $\tau$, the IQE, $f_c$ and the rise time.
2. In the explorer, find the maximum bit rate for $\tau=10$ ns without and with a speed-up capacitor ($k=1$).
3. Design the bias resistor for a white LED ($V_F=3.0$ V at 30 mA) from a 5 V supply. What happens to the current if $V_F$ drops by
   0.2 V when the LED warms up?
4. A pigtailed 1310 nm InGaAsP ELED couples 50 µW into a 9 µm fiber. Compare with the surface-emitter estimate above.